# Using the DynamicMethod Module in baseobjects

## Introduction

DynamicMethod is an abstract method class with multiplexed binding and callback. It inherits DynamicCallable and BaseMethod, with defaults tailored for methods:
- default_bind_method = "bind_self"
- default_call_method = "call_wrapped"
- __call__ forwards self._self_() as the first argument into call_multiplexer

This tutorial highlights how MethodMultiplexer powers both binding and callback in method contexts.

**Prerequisites:**
- Familiarity with BaseCallable and DynamicCallable
- Understanding of the descriptor protocol and method binding

### Table of Contents
- [Importing the Module](#Importing-the-Module)
- [Core Functionality](#Core-Functionality)
- [Module Interaction](#Module-Interaction)
- [Advanced Features](#Advanced-Features)
- [Examples](#Examples)
- [API Highlights](#API-Highlights)
- [Troubleshooting / FAQs](#Troubleshooting--FAQs)
- [Conclusion and Next Steps](#Conclusion-and-Next-Steps)

## Importing the Module

In [1]:
from baseobjects.functions import DynamicMethod

## Core Functionality

DynamicMethod behaves as a descriptor. When accessed via an instance, it binds (by default using bind_self) and its __call__ injects the bound instance.

We'll create a small demo method class that provides alternate call strategies.

In [2]:
class GreeterMethod(DynamicMethod):
    """A DynamicMethod that can greet with different call strategies."""

    def call_wrapped(self, self_obj, name) -> str:
        return f"{self_obj.__class__.__name__} says hello to {name}!"

    def call_with_upper(self, self_obj, name) -> str:
        return f"{self_obj.__class__.__name__} shouts HELLO to {name.upper()}!"


class Person:
    def __init__(self, name) -> None:
        self.name = name

    greet = GreeterMethod()


p = Person("Alice")
print(p.greet("Bob"))  # default call_wrapped

# Switch call method using the multiplexer
Person.greet.call_method = "call_with_upper"
print(p.greet("Bob"))

Person says hello to Bob!
Person shouts HELLO to BOB!


### Method Multiplexer Highlight: Binding and Callback in Methods

- Binding: The bind_multiplexer selects how the descriptor binds when accessed; DynamicMethod defaults to bind_self, appropriate for methods.
- Callback: The call_multiplexer selects which method handles invocation; DynamicMethod.__call__ passes self._self_() (the bound instance) to the selected call strategy.

We can verify the selected methods:

In [3]:
print("Bind method:", Person.greet.bind_method)
print("Call method:", Person.greet.call_method)

Bind method: bind_self
Call method: call_with_upper


## Module Interaction

DynamicMethod integrates with classes as descriptors and works with DynamicFunction as the method_type for dynamic conversion. It inherits all DynamicCallable features for pickling multiplexer state.

In [4]:
import pickle
pickled = pickle.dumps(Person.greet)
restored = pickle.loads(pickled)
print("Restored bind:", restored.bind_method)
print("Restored call:", restored.call_method)

Restored bind: bind_self
Restored call: call_with_upper


## Advanced Features

You can specify call_method during construction or swap at runtime to adapt behavior:

In [5]:
p2 = Person("Charlie")
Person.greet.call_method = "call_wrapped"
print(p2.greet("Dana"))
Person.greet.call_method = "call_with_upper"
print(p2.greet("Dana"))

Person says hello to Dana!
Person shouts HELLO to DANA!


## Examples

- Feature flags: choose different call strategies (validation/logging/transforms) without changing instance code.
- Localization: switch between formatting call methods.

In [6]:
class FriendlyGreeter(DynamicMethod):
    def call_wrapped(self, self_obj, name) -> str:
        return f"Hi {name}, I'm {self_obj.name}."

    def call_formal(self, self_obj, name) -> str:
        return f"Good day, {name}. My name is {self_obj.name}."


class Agent:
    def __init__(self, name) -> None:
        self.name = name
    say = FriendlyGreeter()


a = Agent("Eve")
print(a.say("Frank"))
Agent.say.call_method = "call_formal"
print(a.say("Frank"))

Hi Frank, I'm Eve.
Good day, Frank. My name is Eve.


## API Highlights

- default_bind_method = "bind_self"
- __call__(*args, **kwargs) delegates to call_multiplexer(self._self_(), *args, **kwargs)
- bind_method/call_method properties to select strategies
- Multiplexer-backed pickling of registry and selection

## Troubleshooting / FAQs

### Q: Why is my first argument missing?

A: DynamicMethod.__call__ automatically injects the bound instance as the first argument when invoking the selected call strategy.

### Q: How do I make a different binding strategy?

A: Implement another binding method (e.g., bind_builtin) on your subclass and select it via bind_method.

## Conclusion and Next Steps

DynamicMethod applies MethodMultiplexer to method semantics: binding to instances and selecting among multiple call behaviors. Explore DynamicFunction for function-facing utilities built on this foundation.
